<a href="https://colab.research.google.com/github/HimanshiKaravadra/CODSOFT/blob/main/MovieGenrePrediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/train_data.txt", sep=" ::: ",
    engine="python",
    names=["ID", "TITLE", "GENRE", "PLOT"]
)

In [ ]:
df.head()

,ID,TITLE,GENRE,PLOT
0,1,Oscar et la dame rose (2009),drama,Listening in to a conversation between his doc...
1,2,Cupid (1997),thriller,A brother and sister with a past incestuous re...
2,3,"Young, Wild and Wonderful (1980)",adult,As the bus empties the students for their fiel...
3,4,The Secret Sin (1915),drama,To help their unemployed father make ends meet...
4,5,The Unrecovered (2007),drama,The film's title refers not only to the un-rec...


In [ ]:
df.shape

(54214, 4)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54214 entries, 0 to 54213
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   ID      54214 non-null  int64 
 1   TITLE   54214 non-null  object
 2   GENRE   54214 non-null  object
 3   PLOT    54214 non-null  object
dtypes: int64(1), object(3)
memory usage: 1.7+ MB


In [ ]:
df.isnull().sum()

,0
ID,0
TITLE,0
GENRE,0
PLOT,0


In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z ]', ' ', text)
    words = text.split()
    words = [lemmatizer.lemmatize(word)
             for word in words
             if word not in stop_words]
    return " ".join(words)

In [ ]:
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [ ]:
def clean_text(text):

    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z ]', ' ', text)
    words = text.split()
    words = [lemmatizer.lemmatize(word)
             for word in words
             if word not in stop_words]

    return " ".join(words)

In [ ]:
df["clean_plot"] = df["PLOT"].apply(clean_text)

In [ ]:
df[["PLOT","clean_plot"]].head()

,PLOT,clean_plot
0,Listening in to a conversation between his doc...,listening conversation doctor parent year old ...
1,A brother and sister with a past incestuous re...,brother sister past incestuous relationship cu...
2,As the bus empties the students for their fiel...,bus empty student field trip museum natural hi...
3,To help their unemployed father make ends meet...,help unemployed father make end meet edith twi...
4,The film's title refers not only to the un-rec...,film title refers un recovered body ground zer...


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1,2)
)

X = tfidf.fit_transform(df["clean_plot"])

In [ ]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
y = encoder.fit_transform(df["GENRE"])

In [ ]:
encoder.classes_

array(['action', 'adult', 'adventure', 'animation', 'biography', 'comedy',
       'crime', 'documentary', 'drama', 'family', 'fantasy', 'game-show',
       'history', 'horror', 'music', 'musical', 'mystery', 'news',
       'reality-tv', 'romance', 'sci-fi', 'short', 'sport', 'talk-show',
       'thriller', 'war', 'western'], dtype=object)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
from sklearn.svm import LinearSVC

model = LinearSVC(
    C=1.5,
    class_weight='balanced',
    random_state=42,
    max_iter=5000
)

model.fit(X_train, y_train)
model.fit(X_train, y_train)

LinearSVC(C=1.5, class_weight='balanced', max_iter=5000, random_state=42)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.5109287097666697
              precision    recall  f1-score   support

           0       0.30      0.44      0.36       263
           1       0.38      0.60      0.47       118
           2       0.20      0.28      0.24       155
           3       0.17      0.27      0.21       100
           4       0.03      0.04      0.03        53
           5       0.57      0.51      0.54      1490
           6       0.14      0.18      0.15       101
           7       0.75      0.70      0.73      2619
           8       0.66      0.47      0.55      2723
           9       0.16      0.28      0.20       157
          10       0.08      0.12      0.10        65
          11       0.61      0.64      0.62        39
          12       0.11      0.12      0.11        49
          13       0.52      0.64      0.57       441
          14       0.43      0.64      0.52       146
          15       0.16      0.18      0.17        55
          16       0.05      0.06      0.05        6

In [ ]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred)
print(cm)

[[ 117    1   11    3    0    5    9    5   21    2    7    0    2   12
     1    0    1    0    2    4   13    6    9    0   24    2    6]
 [   0   71   10    0    0    9    2    2    6    0    0    0    0    0
     2    1    0    0    2    2    1    5    1    0    3    0    1]
 [   9    6   44    6    1    9    2   10   14    4    3    0    0   10
     1    1    0    0    7    1    7    4    2    0   11    0    3]
 [   6    0    5   27    0   14    0    3    2   10    5    1    0    5
     0    0    0    0    1    0   10    8    0    0    2    1    0]
 [   1    1    0    0    2    2    1   25    4    2    0    0    0    1
     0    0    1    0    1    0    0    7    1    1    2    0    1]
 [  28   35   23   19    5  765   19   56  170   56   12    3    1   34
    13   13   11    6   31   45   16   64    9   16   32    2    6]
 [  12    3    2    0    0    9   18   11   11    2    0    1    0    5
     0    0    6    0    0    1    0    1    1    0   18    0    0]
 [  34    6   31   1

In [ ]:
prediction = model.predict(new_vector)

NameError: name 'new_vector' is not defined

In [ ]:
predicted_genre = encoder.inverse_transform(prediction)
print("Predicted Genre:", predicted_genre[0])

Predicted Genre: fantasy


In [ ]:
tfidf = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1,2),
    min_df=2,
    max_df=0.9,
    sublinear_tf=True
)

In [ ]:
df["text"] = df["TITLE"] + " " + df["clean_plot"]

In [ ]:
X = tfidf.fit_transform(df["text"])

In [ ]:
# Re-apply TF-IDF transformation with the updated vectorizer
X = tfidf.fit_transform(df["text"])

In [ ]:
# Re-split the data into training and testing sets based on the new X
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
# Retrain the LinearSVC model with the new X_train
from sklearn.svm import LinearSVC

model = LinearSVC(
    C=1.5,
    class_weight='balanced',
    random_state=42,
    max_iter=5000
)

model.fit(X_train, y_train)

LinearSVC(C=1.5, class_weight='balanced', max_iter=5000, random_state=42)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.5594392695748409
              precision    recall  f1-score   support

           0       0.34      0.49      0.41       263
           1       0.48      0.62      0.54       118
           2       0.26      0.29      0.27       155
           3       0.22      0.26      0.24       100
           4       0.00      0.00      0.00        53
           5       0.58      0.56      0.57      1490
           6       0.17      0.17      0.17       101
           7       0.76      0.76      0.76      2619
           8       0.66      0.54      0.59      2723
           9       0.19      0.26      0.22       157
          10       0.13      0.14      0.13        65
          11       0.62      0.67      0.64        39
          12       0.12      0.10      0.11        49
          13       0.58      0.71      0.64       441
          14       0.51      0.66      0.57       146
          15       0.24      0.18      0.21        55
          16       0.14      0.14      0.14        6

In [ ]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred)
print(cm)

[[ 130    1    7    3    0   12    7    5   25    1    4    0    1   10
     0    0    1    0    3    2   13    7    7    0   22    0    2]
 [   1   73    9    0    0    4    0    4    7    1    0    1    0    1
     1    0    0    0    2    1    1    9    0    0    2    0    1]
 [  11    9   45    6    1    9    4    9   13    5    2    0    1    8
     0    1    2    0    7    1    8    3    2    0    6    0    2]
 [   6    0    4   26    0   13    0    4    7   12    4    0    0    5
     0    0    0    0    0    0    8    9    0    0    1    0    1]
 [   1    0    0    0    0    4    1   28    4    1    0    0    1    1
     0    0    0    0    0    0    0    7    1    1    3    0    0]
 [  25   22   14   11    0  840   12   52  217   41   14    4    2   31
    10    6    6    5   27   37   13   54    7   10   26    1    3]
 [  17    0    1    0    0    6   17   10   19    1    0    1    0    5
     0    0    6    1    1    2    0    2    1    0   11    0    0]
 [  18    7   26    

In [ ]:
ngram_range=(1,3)

First, let's fix the `NameError` by importing `joblib`.

In [ ]:
import joblib

In [ ]:
new_plot = """
A young wizard enters a magical school and battles an evil dark lord.
"""

In [ ]:
new_plot = clean_text(new_plot)

In [ ]:
new_vector = tfidf.transform([new_plot])

In [ ]:
prediction = model.predict(new_vector)

In [ ]:
print(encoder.inverse_transform(prediction))

['fantasy']


In [ ]:
import joblib

joblib.dump(model, "genre_model.pkl")
joblib.dump(tfidf, "tfidf_vectorizer.pkl")
joblib.dump(encoder, "label_encoder.pkl")

['label_encoder.pkl']

In [ ]:
model = joblib.load("genre_model.pkl")
tfidf = joblib.load("tfidf_vectorizer.pkl")
encoder = joblib.load("label_encoder.pkl")

In [ ]:
test_df = pd.read_csv(
    "/content/drive/MyDrive/test_data.txt",
    sep=" ::: ",
    engine="python",
    names=["ID", "TITLE", "PLOT"]
)

test_df.head()

,ID,TITLE,PLOT
0,1,Edgar's Lunch (1998),"L.R. Brane loves his life - his car, his apart..."
1,2,La guerra de papá (1977),"Spain, March 1964: Quico is a very naughty chi..."
2,3,Off the Beaten Track (2010),One year in the life of Albin and his family o...
3,4,Meu Amigo Hindu (2015),"His father has died, he hasn't spoken with his..."
4,5,Er nu zhai (1955),Before he was known internationally as a marti...


In [ ]:
test_df["clean_plot"] = test_df["PLOT"].apply(clean_text)

In [ ]:
X_test_final = tfidf.transform(test_df["clean_plot"])

In [ ]:
tfidf.transform(test_df["clean_plot"])

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 2415367 stored elements and shape (54200, 10000)>

In [ ]:
pred = model.predict(X_test_final)

In [ ]:
predicted_genres = encoder.inverse_transform(pred)

In [ ]:
test_df["Predicted_Genre"] = predicted_genres

In [ ]:
test_df[["TITLE", "Predicted_Genre"]].head()

,TITLE,Predicted_Genre
0,Edgar's Lunch (1998),comedy
1,La guerra de papá (1977),drama
2,Off the Beaten Track (2010),documentary
3,Meu Amigo Hindu (2015),drama
4,Er nu zhai (1955),action


In [ ]:
solution_df = pd.read_csv(
    "/content/drive/MyDrive/test_data_solution.txt",
    sep=" ::: ",
    engine="python",
    names=["ID", "TITLE", "GENRE", "PLOT"]
)

solution_df.head()

,ID,TITLE,GENRE,PLOT
0,1,Edgar's Lunch (1998),thriller,"L.R. Brane loves his life - his car, his apart..."
1,2,La guerra de papá (1977),comedy,"Spain, March 1964: Quico is a very naughty chi..."
2,3,Off the Beaten Track (2010),documentary,One year in the life of Albin and his family o...
3,4,Meu Amigo Hindu (2015),drama,"His father has died, he hasn't spoken with his..."
4,5,Er nu zhai (1955),drama,Before he was known internationally as a marti...


In [ ]:
y_true = encoder.transform(solution_df["GENRE"])

In [ ]:
y_pred = encoder.transform(test_df["Predicted_Genre"])

In [ ]:
from sklearn.metrics import accuracy_score
print("Test Accuracy:", accuracy_score(y_true, y_pred))

Test Accuracy: 0.509760147601476


In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

           0       0.30      0.42      0.35      1314
           1       0.37      0.52      0.44       590
           2       0.21      0.29      0.24       775
           3       0.16      0.22      0.19       498
           4       0.04      0.05      0.04       264
           5       0.57      0.50      0.53      7446
           6       0.12      0.20      0.15       505
           7       0.75      0.71      0.73     13096
           8       0.66      0.49      0.56     13612
           9       0.16      0.26      0.20       783
          10       0.13      0.17      0.15       322
          11       0.68      0.70      0.69       193
          12       0.07      0.09      0.08       243
          13       0.51      0.64      0.57      2204
          14       0.44      0.62      0.51       731
          15       0.10      0.16      0.12       276
          16       0.08      0.10      0.09       318
          17       0.22    

In [ ]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_true, y_pred)
print(cm)

[[ 551    9   53   22    2   63   56   44  107   15   13    1   15   51
     3    8    7    3   11   18   70   35   31    4   87   11   24]
 [   7  309   28    2    0   59   12   10   42    5    2    2    1   16
     3    2    3    0    8    4    9   38    8    0   18    0    2]
 [  61   46  225   27    5   51   14   55   60   26   18    1   10   30
     4    1   10    2   20    6   31   26    7    2   19    1   17]
 [  30    0   32  111    2   58    2   31   28   52   25    0    4   24
     4    4    2    3   12    2   21   40    2    3    5    1    0]
 [   5    0   10    0   13    9    2  120   33    4    0    0    6    2
     9    3    0    0    7    3    1   16    7    5    4    2    3]
 [ 155  129  122  110   26 3710  102  233  935  214   44   16   23  201
    63   56   46   20  178  249   63  395   24   55  210   14   53]
 [  63    3   10    2    2   40   99   25   76    6    2    0    2   19
     3    1   24    1    4    6    5   19    1    0   85    0    7]
 [ 131   48  138   6